# 🚀 Модуль 2. Урок 5. Оконные функции (Window Functions): ранжирование, кумулятивные итоги и сглаживание

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_05_window_functions_advanced.ipynb)

## 📌 Бизнес-контекст и методология
Оконные функции (`OVER (PARTITION BY ... ORDER BY ...)`) — один из самых мощных инструментов продуктового аналитика.
В отличие от обычного `GROUP BY`, оконные функции **не схлопывают строки**, а возвращают расчет для каждой строки с учетом контекста окна.
Применение в реальном бизнесе:
- Сглаживание сезонных всплесков и выходных (Rolling 7-day Moving Average)
- Накопительный итог выручки к цели квартала (Cumulative Run-rate)
- Сравнение с предыдущим периодом (MoM growth через `LAG` / `LEAD`)
- Выделение топ-товаров внутри категорий (`ROW_NUMBER`, `DENSE_RANK`)

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.edgecolor'] = '#e6e6e6'
plt.rcParams['axes.linewidth'] = 0.8

PRIMARY = "#1779ba"
SUCCESS = "#3adb76"
ALERT = "#cc4b37"
WARNING = "#ffae00"
SECONDARY = "#767676"

def ensure_dataset(filename, subfolder=""):
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    urllib.request.urlretrieve(url, filename)
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")

🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных
Рассчитаем ежедневную выручку интернет-магазина, скользящее среднее за 7 дней для сглаживания шума выходных и накопительную выручку с начала года.

In [2]:
# Демонстрация: Скользящее среднее и кумулятивный итог
query_window = """
WITH daily_revenue AS (
    SELECT 
        DATE(o.order_date) as order_day,
        ROUND(SUM(oi.total_amount), 2) as daily_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY DATE(o.order_date)
)
SELECT 
    order_day,
    daily_total,
    ROUND(AVG(daily_total) OVER (
        ORDER BY order_day 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) as rolling_7d_avg,
    ROUND(SUM(daily_total) OVER (
        ORDER BY order_day
    ), 2) as cumulative_revenue
FROM daily_revenue
ORDER BY order_day;
"""
df_window = pd.read_sql_query(query_window, conn)
display(df_window.tail(8))

,order_day,daily_total,rolling_7d_avg,cumulative_revenue
556,2026-09-15,919.44,1847.01,2600211.91
557,2026-09-17,2219.12,1640.79,2602431.03
558,2026-09-22,1667.88,1667.23,2604098.91
559,2026-09-24,1942.35,1905.55,2606041.26
560,2026-09-25,1195.36,1625.23,2607236.62
561,2026-09-28,1837.99,1848.65,2609074.61
562,2026-09-29,265.72,1435.41,2609340.33
563,2026-10-01,1019.28,1449.67,2610359.61


## 📊 Интерактивная анимация: Динамика выручки и 7-дневное сглаживание
Посмотрим, как скользящее окно фильтрует шум ежедневных флуктуаций и формирует истинный бизнес-тренд выручки.

In [3]:
# Построение анимации динамики выручки во времени
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), dpi=105, sharex=True)

df_sample = df_window.tail(90).reset_index(drop=True)
dates = pd.to_datetime(df_sample['order_day'])

def update(frame):
    ax1.clear()
    ax2.clear()
    
    sub = df_sample.iloc[:frame+1]
    sub_dates = dates[:frame+1]
    
    # График 1: Daily vs 7-day Rolling
    ax1.plot(sub_dates, sub['daily_total'], color='#dcdcdc', alpha=0.8, linestyle='--', label='Сырая дневная выручка')
    ax1.plot(sub_dates, sub['rolling_7d_avg'], color=PRIMARY, linewidth=2.5, label='7-Day Rolling Average')
    
    if len(sub) > 0:
        latest_val = sub['rolling_7d_avg'].iloc[-1]
        ax1.scatter([sub_dates.iloc[-1]], [latest_val], color=ALERT, s=50, zorder=5)
        ax1.annotate(f"${latest_val:.0f}", (sub_dates.iloc[-1], latest_val), 
                     textcoords="offset points", xytext=(0, 8), ha='center', fontweight='bold', color=ALERT)
        
    ax1.set_title("Сглаживание шума: Дневная выручка vs 7-дневное скользящее среднее", fontsize=11, fontweight='bold')
    ax1.set_ylabel("Выручка в день ($)")
    ax1.set_xlim(dates.min(), dates.max())
    ax1.set_ylim(0, df_sample['daily_total'].max() * 1.25)
    ax1.legend(loc='upper left', frameon=True, facecolor='#ffffff')
    
    # График 2: Cumulative Run-Rate
    ax2.plot(sub_dates, sub['cumulative_revenue'] / 1000, color=SUCCESS, linewidth=2.2, label='Кумулятивная выручка ($k)')
    ax2.fill_between(sub_dates, 0, sub['cumulative_revenue'] / 1000, color=SUCCESS, alpha=0.15)
    
    ax2.set_title("Кумулятивная выручка (Cumulative Run-Rate, $ тыс.)", fontsize=11, fontweight='bold')
    ax2.set_ylabel("Накопительный итог ($ тыс.)")
    ax2.set_xlabel("Дата транзакции")
    ax2.set_ylim(0, df_sample['cumulative_revenue'].max() / 1000 * 1.15)
    ax2.legend(loc='upper left', frameon=True, facecolor='#ffffff')
    
    fig.autofmt_xdate()

# Создаем анимацию на 25 кадров
anim = FuncAnimation(fig, update, frames=np.linspace(5, len(df_sample)-1, 25, dtype=int), interval=160)
plt.close(fig)
HTML(anim.to_jshtml())

### 🎯 Практическое задание 1: Поиск второго чека клиента через DENSE_RANK
Для каждого клиента пронумеруйте его завершенные заказы по сумме чека в порядке убывания и выведите второй по величине чек с помощью `DENSE_RANK()`.

In [4]:
### ВАШ КОД ЗДЕСЬ
query_rank = """
WITH ranked_orders AS (
    SELECT 
        o.customer_id,
        o.order_id,
        o.order_date,
        SUM(oi.total_amount) as order_amount,
        DENSE_RANK() OVER (
            PARTITION BY o.customer_id 
            ORDER BY SUM(oi.total_amount) DESC
        ) as amount_rank
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY o.customer_id, o.order_id, o.order_date
)
SELECT 
    customer_id,
    order_id,
    ROUND(order_amount, 2) as second_largest_amount
FROM ranked_orders
WHERE amount_rank = 2
LIMIT 8;
"""

df_rank = pd.read_sql_query(query_rank, conn)
display(df_rank.head(5))

assert not df_rank.empty, "Должны быть клиенты со вторым чеком"
print("✅ Тест успешно пройден: оконная функция DENSE_RANK отработала безупречно!")

,customer_id,order_id,second_largest_amount
0,1,10002,1236.36
1,4,10006,3409.93
2,9,10015,2419.25
3,10,10021,997.84
4,11,10023,932.51


✅ Тест успешно пройден: оконная функция DENSE_RANK отработала безупречно!


### 💡 Разница между ROW_NUMBER, RANK и DENSE_RANK
- `ROW_NUMBER()`: строго уникальный номер строки (1, 2, 3, 4)
- `RANK()`: при совпадении значений ставит одинаковый ранг с пропуском последующих (1, 2, 2, 4)
- `DENSE_RANK()`: при совпадении значений ставит одинаковый ранг **без** пропусков (1, 2, 2, 3)